# Data Preprocessing

In [94]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.append(os.path.abspath(".."))

In [95]:
from src.utils.utils import ConfigManager
import pandas as pd 
import numpy as np


In [96]:
# Usage configuration manager
config_manager = ConfigManager('config/config.yaml')
# Load the config
config = config_manager.load_config()
config

{'features': {'cat_nominal': ['person_home_ownership', 'loan_intent'],
  'cat_ordinal': {'cb_person_default_on_file': {'N': 0, 'Y': 1},
   'loan_grade': {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}},
  'categorical': ['person_home_ownership',
   'loan_intent',
   'loan_grade',
   'cb_person_default_on_file'],
  'numerical': ['person_age',
   'person_income',
   'person_emp_length',
   'loan_amnt',
   'loan_int_rate',
   'loan_percent_income',
   'cb_person_cred_hist_length']},
 'path': {'interim_data': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/data/interim',
  'logrob_preprocessing': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/models/logrob_prep_pipeline.pkl',
  'models': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/models',
  'path_config': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/config/config.yaml',
  'processed_data'

In [97]:
# Data path
train_data_path = config['path']['train_data']
valid_data_path = config['path']['valid_data']
test_data_path = config['path']['test_data']
# Deserialization
data_train = config_manager.deserialize_data(train_data_path)
data_valid = config_manager.deserialize_data(valid_data_path)
data_test = config_manager.deserialize_data(test_data_path)

In [98]:
# Split the data
X_train, y_train = data_train['X_train'], data_train['y_train']
X_valid, y_valid = data_valid['X_valid'], data_valid['y_valid']
X_test, y_test = data_test['X_test'], data_test['y_test']

In [99]:
X_train

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
352,23,16000,MORTGAGE,3.0,DEBTCONSOLIDATION,C,2275,13.06,0.14,Y,4
12819,23,85000,MORTGAGE,7.0,VENTURE,B,25000,10.36,0.29,N,4
19720,33,35360,MORTGAGE,7.0,MEDICAL,A,6350,7.51,0.18,N,5
27174,29,125000,MORTGAGE,12.0,PERSONAL,B,25000,10.62,0.20,N,8
8274,24,57000,MORTGAGE,0.0,MEDICAL,B,2400,12.69,0.04,N,4
...,...,...,...,...,...,...,...,...,...,...,...
29808,43,39000,MORTGAGE,12.0,VENTURE,B,8000,9.99,0.21,N,16
5396,24,45000,MORTGAGE,8.0,MEDICAL,A,5000,5.79,0.11,N,3
866,21,23000,MORTGAGE,6.0,DEBTCONSOLIDATION,C,3000,13.35,0.13,N,3
15801,24,79200,RENT,3.0,EDUCATION,D,16000,12.67,0.20,Y,2


# Impute mising value with median

In [100]:
def num_imputer(data: pd.DataFrame, cols: list) -> object:
    from sklearn.impute import SimpleImputer
    num_imputer = SimpleImputer(strategy='median', missing_values=np.nan)
    num_imputer.fit(data[cols])
    return num_imputer

def num_imputer_transform(data: pd.DataFrame, cols: list, imputer: object) -> pd.DataFrame:
    data = data.copy()
    imputed_data = imputer.transform(data[cols])
    result_df = pd.DataFrame(
        data=imputed_data, index=data.index, columns=cols
    )
    # drop data
    data.drop(cols, axis=1, inplace=True)
    # hasil
    data = pd.concat([data, result_df], axis=1)
    return data

In [101]:
# Get numerical features from config
num_cols = config['features']['numerical']
# fit the imputer
imputer = num_imputer(data=X_train, cols=num_cols)
# transform the imputer
X_train = num_imputer_transform(data=X_train, imputer=imputer, cols=num_cols)
X_valid = num_imputer_transform(data=X_valid, imputer=imputer, cols=num_cols)
X_test = num_imputer_transform(data=X_test, imputer=imputer, cols=num_cols)

In [102]:
X_train.head(3)

,person_home_ownership,loan_intent,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
352,MORTGAGE,DEBTCONSOLIDATION,C,Y,23.0,16000.0,3.0,2275.0,13.06,0.14,4.0
12819,MORTGAGE,VENTURE,B,N,23.0,85000.0,7.0,25000.0,10.36,0.29,4.0
19720,MORTGAGE,MEDICAL,A,N,33.0,35360.0,7.0,6350.0,7.51,0.18,5.0


# Encoding data categorical

## Label Encoding

In [103]:
# get the columns for encoding from config
cat_nom_cols = config['features']['cat_nominal']
cat_ord_map = config['features']['cat_ordinal']
print(cat_nom_cols)
print(cat_ord_map)

['person_home_ownership', 'loan_intent']
{'cb_person_default_on_file': {'N': 0, 'Y': 1}, 'loan_grade': {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}}


In [104]:
# Function to mapping ordinal categorical features
def ordinal_mapper(X: pd.DataFrame, mapping: dict) -> pd.DataFrame:
    X = X.copy()
    
    for col, map_dict in mapping.items():
        # Defense if its already numeric
        if pd.api.types.is_numeric_dtype(X[col]):
            continue
        X[col] = X[col].map(map_dict)
        
    return X

In [105]:
# implementation to X
X_train = ordinal_mapper(X=X_train, mapping=cat_ord_map)
X_valid = ordinal_mapper(X=X_valid, mapping=cat_ord_map)
X_test = ordinal_mapper(X=X_test, mapping=cat_ord_map)

# check the result
display(
    X_train[list(cat_ord_map.keys())].head(5),
    X_valid.head(3),
    X_test.head(3)
)

,cb_person_default_on_file,loan_grade
352,1,3
12819,0,2
19720,0,1
27174,0,2
8274,0,2


,person_home_ownership,loan_intent,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
23144,MORTGAGE,EDUCATION,3,1,27.0,60000.0,3.0,5000.0,12.73,0.08,8.0
20604,RENT,VENTURE,3,1,32.0,128000.0,11.0,10000.0,13.85,0.08,5.0
823,OWN,VENTURE,5,1,25.0,22000.0,0.0,7500.0,10.99,0.34,3.0


,person_home_ownership,loan_intent,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
25333,RENT,EDUCATION,3,1,28.0,25000.0,2.0,3000.0,13.22,0.12,8.0
14531,MORTGAGE,EDUCATION,1,0,25.0,109000.0,9.0,14000.0,9.63,0.13,3.0
18399,RENT,HOMEIMPROVEMENT,2,0,27.0,75000.0,1.0,20000.0,10.99,0.27,6.0


# One-hot Encoding

In [106]:
def fit_ohe_encoder(data: pd.DataFrame, cols: list) -> object:
    from sklearn.preprocessing import OneHotEncoder
    ohe = OneHotEncoder(handle_unknown='ignore', drop='first',sparse_output=False)
    ohe.fit(data[cols])
    return ohe 

def transform_ohe(data: pd.DataFrame, ohe: object, cols:list) -> pd.DataFrame:
    data = data.copy()
    #transformed data
    transformed = ohe.transform(data[cols])
    # get features name
    col_names = ohe.get_feature_names_out(cols)
    # convert to dataframe
    ohe_df = pd.DataFrame(transformed, columns=col_names, index=data.index)
    # drop the original data
    data.drop(cols, axis=1, inplace=True)
    # concat data
    data = pd.concat([data, ohe_df], axis=1)
    return data

In [107]:
ohe = fit_ohe_encoder(X_train, cols=cat_nom_cols)

X_train = transform_ohe(data=X_train, ohe=ohe, cols=cat_nom_cols)
X_valid = transform_ohe(data=X_valid, ohe=ohe, cols=cat_nom_cols)
X_test = transform_ohe(data=X_test, ohe=ohe, cols=cat_nom_cols)

In [108]:
display(
    X_train.head(3),
    X_valid.head(3),
    X_test.head(3)
)

,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
352,3,1,23.0,16000.0,3.0,2275.0,13.06,0.14,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12819,2,0,23.0,85000.0,7.0,25000.0,10.36,0.29,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19720,1,0,33.0,35360.0,7.0,6350.0,7.51,0.18,5.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
23144,3,1,27.0,60000.0,3.0,5000.0,12.73,0.08,8.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
20604,3,1,32.0,128000.0,11.0,10000.0,13.85,0.08,5.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
823,5,1,25.0,22000.0,0.0,7500.0,10.99,0.34,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
25333,3,1,28.0,25000.0,2.0,3000.0,13.22,0.12,8.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
14531,1,0,25.0,109000.0,9.0,14000.0,9.63,0.13,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
18399,2,0,27.0,75000.0,1.0,20000.0,10.99,0.27,6.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


# Scaling Data


Scaling menggunakan robust scaler, akan terdapat 2 model features untuk dilakukan eksperimen, features dengan robust scaler dan features dengan log transform + robust scaler.

## Robust Scaler

In [109]:
def fit_robust_scaler(data: pd.DataFrame, cols: list, is_log: bool = False) -> object:
    from sklearn.preprocessing import RobustScaler
    data = data.copy()
    if is_log:
        data[cols] = np.log1p(data[cols])

    scaler = RobustScaler()
    scaler.fit(data[cols])
    return scaler

def transform_scaler(data: pd.DataFrame, scaler: object, cols: list, is_log: bool = False) -> pd.DataFrame:
    data = data.copy()

    if is_log:
        data[cols] = np.log1p(data[cols])
        
    data[cols] = scaler.transform(data[cols])
    return data

In [110]:
# Scaler only
scaler_rob = fit_robust_scaler(data=X_train, cols=num_cols)

# Transform features with robust scaler
X_train_rob = transform_scaler(data=X_train, scaler=scaler_rob, cols=num_cols)
X_valid_rob = transform_scaler(data=X_valid, scaler=scaler_rob, cols=num_cols)
X_test_rob = transform_scaler(data=X_test, scaler=scaler_rob, cols=num_cols)

In [111]:
# Check Result
display(X_train_rob.head(), X_valid_rob.head(), X_test_rob.head())

,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
352,3,1,-0.428571,-0.959764,-0.2,-0.795139,0.448052,-0.071429,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12819,2,0,-0.428571,0.738280,0.6,2.361111,-0.136364,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19720,1,0,1.000000,-0.483327,0.6,-0.229167,-0.753247,0.214286,0.2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
27174,2,0,0.428571,1.722653,1.6,2.361111,-0.080087,0.357143,0.8,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8274,2,0,-0.285714,0.049219,-0.8,-0.777778,0.367965,-0.785714,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
23144,3,1,0.142857,0.123047,-0.2,-0.416667,0.376623,-0.500000,0.8,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
20604,3,1,0.857143,1.796481,1.4,0.277778,0.619048,-0.500000,0.2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
823,5,1,-0.142857,-0.812108,-0.8,-0.069444,0.000000,1.357143,-0.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
17023,1,0,-0.571429,2.337886,-0.4,0.027778,-0.865801,-0.714286,-0.2,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
7704,2,0,-0.714286,-0.023723,-0.2,0.000000,0.080087,0.000000,-0.4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
25333,3,1,0.285714,-0.738280,-0.4,-0.694444,0.482684,-0.214286,0.8,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
14531,1,0,-0.142857,1.328904,1.0,0.833333,-0.294372,-0.142857,-0.2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
18399,2,0,0.142857,0.492187,-0.6,1.666667,0.000000,0.857143,0.4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
21053,1,0,1.285714,-0.246093,0.0,-0.069444,-0.800866,0.142857,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22172,3,1,0.714286,-0.393749,0.2,-0.194444,0.606061,0.142857,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [112]:
# Check using method info
X_train_rob.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19544 entries, 352 to 23660
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_grade                   19544 non-null  int64  
 1   cb_person_default_on_file    19544 non-null  int64  
 2   person_age                   19544 non-null  float64
 3   person_income                19544 non-null  float64
 4   person_emp_length            19544 non-null  float64
 5   loan_amnt                    19544 non-null  float64
 6   loan_int_rate                19544 non-null  float64
 7   loan_percent_income          19544 non-null  float64
 8   cb_person_cred_hist_length   19544 non-null  float64
 9   person_home_ownership_OTHER  19544 non-null  float64
 10  person_home_ownership_OWN    19544 non-null  float64
 11  person_home_ownership_RENT   19544 non-null  float64
 12  loan_intent_EDUCATION        19544 non-null  float64
 13  loan_intent_HOMEIMP

In [113]:
X_valid_rob.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6515 entries, 23144 to 9369
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_grade                   6515 non-null   int64  
 1   cb_person_default_on_file    6515 non-null   int64  
 2   person_age                   6515 non-null   float64
 3   person_income                6515 non-null   float64
 4   person_emp_length            6515 non-null   float64
 5   loan_amnt                    6515 non-null   float64
 6   loan_int_rate                6515 non-null   float64
 7   loan_percent_income          6515 non-null   float64
 8   cb_person_cred_hist_length   6515 non-null   float64
 9   person_home_ownership_OTHER  6515 non-null   float64
 10  person_home_ownership_OWN    6515 non-null   float64
 11  person_home_ownership_RENT   6515 non-null   float64
 12  loan_intent_EDUCATION        6515 non-null   float64
 13  loan_intent_HOMEIMP

In [114]:
X_test_rob.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6515 entries, 25333 to 21409
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_grade                   6515 non-null   int64  
 1   cb_person_default_on_file    6515 non-null   int64  
 2   person_age                   6515 non-null   float64
 3   person_income                6515 non-null   float64
 4   person_emp_length            6515 non-null   float64
 5   loan_amnt                    6515 non-null   float64
 6   loan_int_rate                6515 non-null   float64
 7   loan_percent_income          6515 non-null   float64
 8   cb_person_cred_hist_length   6515 non-null   float64
 9   person_home_ownership_OTHER  6515 non-null   float64
 10  person_home_ownership_OWN    6515 non-null   float64
 11  person_home_ownership_RENT   6515 non-null   float64
 12  loan_intent_EDUCATION        6515 non-null   float64
 13  loan_intent_HOMEIM

## Log Transform + Robust

In [115]:
# Log + Scaler 
log_scaler = fit_robust_scaler(data=X_train, cols=num_cols, is_log=True)

# Transform features with robust scaler
X_train_logrob = transform_scaler(data=X_train, scaler=log_scaler, cols=num_cols, is_log=True)
X_valid_logrob = transform_scaler(data=X_valid, scaler=log_scaler, cols=num_cols, is_log=True)
X_test_logrob = transform_scaler(data=X_test, scaler=log_scaler, cols=num_cols, is_log=True)

In [116]:
# Check Result
display(X_train_logrob.head(), X_valid_logrob.head(), X_test_logrob.head())

,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
352,3,1,-0.460210,-1.715772,-0.227505,-1.409547,0.401520,-0.072277,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12819,2,0,-0.460210,0.604921,0.479190,1.277469,-0.136078,0.950709,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19720,1,0,0.900717,-0.613856,0.479190,-0.258952,-0.864327,0.213119,0.22483,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
27174,2,0,0.411672,1.140842,0.974187,1.277469,-0.079026,0.352208,0.72483,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8274,2,0,-0.300707,0.049634,-1.640895,-1.349600,0.334285,-0.832044,0.00000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
23144,3,1,0.142098,0.120911,-0.227505,-0.526897,0.341641,-0.519718,0.724830,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
20604,3,1,0.784074,1.173799,0.892580,0.250167,0.539341,-0.519718,0.224830,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
823,5,1,-0.147462,-1.273264,-1.640895,-0.072353,0.000000,1.265410,-0.275170,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
17023,1,0,-0.626502,1.394199,-0.520810,0.027683,-1.023288,-0.752850,-0.275170,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
7704,2,0,-0.800186,-0.024572,-0.227505,0.000000,0.076624,0.000000,-0.629926,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
25333,3,1,0.279209,-1.095631,-0.520810,-1.099499,0.430048,-0.218752,0.724830,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
14531,1,0,-0.147462,0.950512,0.706695,0.627396,-0.303528,-0.145191,-0.275170,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
18399,2,0,0.142098,0.430993,-0.934200,1.027286,0.000000,0.821399,0.414921,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
21053,1,0,1.124051,-0.278852,0.000000,-0.072353,-0.930361,0.142687,0.854756,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22172,3,1,0.663841,-0.477704,0.185885,-0.215663,0.529134,0.142687,0.854756,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [117]:
X_valid_logrob.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6515 entries, 23144 to 9369
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_grade                   6515 non-null   int64  
 1   cb_person_default_on_file    6515 non-null   int64  
 2   person_age                   6515 non-null   float64
 3   person_income                6515 non-null   float64
 4   person_emp_length            6515 non-null   float64
 5   loan_amnt                    6515 non-null   float64
 6   loan_int_rate                6515 non-null   float64
 7   loan_percent_income          6515 non-null   float64
 8   cb_person_cred_hist_length   6515 non-null   float64
 9   person_home_ownership_OTHER  6515 non-null   float64
 10  person_home_ownership_OWN    6515 non-null   float64
 11  person_home_ownership_RENT   6515 non-null   float64
 12  loan_intent_EDUCATION        6515 non-null   float64
 13  loan_intent_HOMEIMP

In [118]:
# 
display(
    X_train_rob.describe(),
    X_train_logrob.describe()
)

,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
count,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000
mean,2.221193,0.176627,0.248845,0.263778,0.154758,0.218897,0.008089,0.144008,0.364183,0.003428,0.078797,0.503121,0.200778,0.110827,0.181181,0.170538,0.174325
std,1.167118,0.381363,0.885952,1.238320,0.800405,0.878286,0.664985,0.762639,0.812708,0.058452,0.269428,0.500003,0.400592,0.313926,0.385178,0.376115,0.379398
min,1.000000,0.000000,-0.857143,-1.255076,-0.800000,-1.041667,-1.205628,-1.071429,-0.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,-0.428571,-0.404454,-0.400000,-0.416667,-0.541126,-0.428571,-0.200000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,3.000000,0.000000,0.571429,0.595546,0.600000,0.583333,0.458874,0.571429,0.800000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,7.000000,1.000000,9.714286,45.404208,7.400000,3.750000,2.647186,4.500000,5.200000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
count,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000,19544.000000
mean,2.221193,0.176627,0.167976,0.012736,-0.122893,-0.052202,-0.078004,0.111098,0.202213,0.003428,0.078797,0.503121,0.200778,0.110827,0.181181,0.170538,0.174325
std,1.167118,0.381363,0.743337,0.785858,0.796210,0.795209,0.668399,0.727282,0.646927,0.058452,0.269428,0.500003,0.400592,0.313926,0.385178,0.376115,0.379398
min,1.000000,0.000000,-0.981953,-3.641943,-1.640895,-3.106601,-1.574846,-1.156621,-0.629926,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,-0.460210,-0.493290,-0.520810,-0.526897,-0.589530,-0.443444,-0.275170,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,3.000000,0.000000,0.539790,0.506710,0.479190,0.473103,0.410470,0.556556,0.724830,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,7.000000,1.000000,4.915498,4.922416,2.169829,1.654718,1.772632,3.615228,2.249946,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# Serialization Data

In [119]:
data_train = {
    'Robust Scaler' : {
        'X_train' : X_train_rob,
        'y_train' : y_train
    },
    'Log + Robust Scaler' : {
        'X_train' : X_train_logrob,
        'y_train' : y_train
    }
}

data_valid = {
    'Robust Scaler' : {
        'X_valid' : X_valid_rob,
        'y_valid' : y_valid
    },
    'Log + Robust Scaler' : {
        'X_valid' : X_valid_logrob,
        'y_valid' : y_valid
    }
}

data_test = {
    'Robust Scaler' : {
        'X_test' : X_test_rob,
        'y_test' : y_test
    },
    'Log + Robust Scaler' : {
        'X_test' : X_test_logrob,
        'y_test' : y_test
    }
}

PATH_PROCESSED = config['path']['processed_data']
path_train = f"{PATH_PROCESSED}/train.pkl"
path_valid = f"{PATH_PROCESSED}/valid.pkl"
path_test = f"{PATH_PROCESSED}/test.pkl"


config_manager.serialized_data(data_train, path_train)
config_manager.serialized_data(data_valid, path_valid)
config_manager.serialized_data(data_test, path_test)

In [120]:
config_manager.update_config('path.train_data_processed', path_train)
config_manager.update_config('path.valid_data_processed', path_valid)
config_manager.update_config('path.test_data_processed', path_test)

# Validation from Pipeline with Notebooks

In [121]:
# Data path
train_data_path = config['path']['train_data_processed_pipe']
valid_data_path = config['path']['valid_data_processed_pipe']
test_data_path = config['path']['test_data_processed_pipe']
# Deserialization
data_train = config_manager.deserialize_data(train_data_path)
data_valid = config_manager.deserialize_data(valid_data_path)
data_test = config_manager.deserialize_data(test_data_path)

In [122]:
X_train_logrob1 = data_train['Log + Robust Scaler']['X_train']
X_train_logrob1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19544 entries, 352 to 23660
Data columns (total 17 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   num__person_age                       19544 non-null  float64
 1   num__person_income                    19544 non-null  float64
 2   num__person_emp_length                19544 non-null  float64
 3   num__loan_amnt                        19544 non-null  float64
 4   num__loan_int_rate                    19544 non-null  float64
 5   num__loan_percent_income              19544 non-null  float64
 6   num__cb_person_cred_hist_length       19544 non-null  float64
 7   cat_nom__person_home_ownership_OTHER  19544 non-null  float64
 8   cat_nom__person_home_ownership_OWN    19544 non-null  float64
 9   cat_nom__person_home_ownership_RENT   19544 non-null  float64
 10  cat_nom__loan_intent_EDUCATION        19544 non-null  float64
 11  cat_nom__loan_inte

In [123]:
X_train_logrob.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19544 entries, 352 to 23660
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_grade                   19544 non-null  int64  
 1   cb_person_default_on_file    19544 non-null  int64  
 2   person_age                   19544 non-null  float64
 3   person_income                19544 non-null  float64
 4   person_emp_length            19544 non-null  float64
 5   loan_amnt                    19544 non-null  float64
 6   loan_int_rate                19544 non-null  float64
 7   loan_percent_income          19544 non-null  float64
 8   cb_person_cred_hist_length   19544 non-null  float64
 9   person_home_ownership_OTHER  19544 non-null  float64
 10  person_home_ownership_OWN    19544 non-null  float64
 11  person_home_ownership_RENT   19544 non-null  float64
 12  loan_intent_EDUCATION        19544 non-null  float64
 13  loan_intent_HOMEIMP

In [124]:
X_train_logrob1.columns = [col.split("__")[-1] for col in X_train_logrob1.columns]

In [125]:
X_train_logrob1.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_EDUCATION',
       'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
       'loan_intent_PERSONAL', 'loan_intent_VENTURE',
       'cb_person_default_on_file', 'loan_grade'],
      dtype='object')

In [126]:
X_train_logrob.columns

Index(['loan_grade', 'cb_person_default_on_file', 'person_age',
       'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate',
       'loan_percent_income', 'cb_person_cred_hist_length',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_EDUCATION',
       'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
       'loan_intent_PERSONAL', 'loan_intent_VENTURE'],
      dtype='object')

In [127]:
col_notebooks = X_train_logrob.columns.to_list()
col_notebooks

['loan_grade',
 'cb_person_default_on_file',
 'person_age',
 'person_income',
 'person_emp_length',
 'loan_amnt',
 'loan_int_rate',
 'loan_percent_income',
 'cb_person_cred_hist_length',
 'person_home_ownership_OTHER',
 'person_home_ownership_OWN',
 'person_home_ownership_RENT',
 'loan_intent_EDUCATION',
 'loan_intent_HOMEIMPROVEMENT',
 'loan_intent_MEDICAL',
 'loan_intent_PERSONAL',
 'loan_intent_VENTURE']

In [128]:
X_train_logrob1 = X_train_logrob1[col_notebooks]
X_train_logrob1.columns

Index(['loan_grade', 'cb_person_default_on_file', 'person_age',
       'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate',
       'loan_percent_income', 'cb_person_cred_hist_length',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_EDUCATION',
       'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
       'loan_intent_PERSONAL', 'loan_intent_VENTURE'],
      dtype='object')

In [129]:
# Check is same values between data with pipeline and notebooks
(X_train_logrob1 == X_train_logrob).all().all()

np.True_

In [130]:
display(X_train_logrob1.head(), X_train_logrob.head())

,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
352,3,1,-0.460210,-1.715772,-0.227505,-1.409547,0.401520,-0.072277,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12819,2,0,-0.460210,0.604921,0.479190,1.277469,-0.136078,0.950709,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19720,1,0,0.900717,-0.613856,0.479190,-0.258952,-0.864327,0.213119,0.22483,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
27174,2,0,0.411672,1.140842,0.974187,1.277469,-0.079026,0.352208,0.72483,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8274,2,0,-0.300707,0.049634,-1.640895,-1.349600,0.334285,-0.832044,0.00000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,loan_grade,cb_person_default_on_file,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
352,3,1,-0.460210,-1.715772,-0.227505,-1.409547,0.401520,-0.072277,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12819,2,0,-0.460210,0.604921,0.479190,1.277469,-0.136078,0.950709,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19720,1,0,0.900717,-0.613856,0.479190,-0.258952,-0.864327,0.213119,0.22483,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
27174,2,0,0.411672,1.140842,0.974187,1.277469,-0.079026,0.352208,0.72483,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8274,2,0,-0.300707,0.049634,-1.640895,-1.349600,0.334285,-0.832044,0.00000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
